# 01. Exploratory Data Analysis (EDA)
## Water Quality Prediction via Gradient Boosting & Deep Neural Networks (Phase 1)

### Objectives:
- Load and inspect the CCME Water Quality Telemetry dataset (`data/raw/Combined_dataset.csv`).
- Analyze distributions of primary physical and chemical sensors (pH, DO, BOD, Ammonia, Nitrate, Nitrogen, Orthophosphate, Temperature).
- Examine environmental context (Waterbody Type, Monitoring Site Area, Temporal range).
- Evaluate target distributions: Continuous `CCME_Values` (0-100 index) and Discrete `CCME_WQI` (Excellent, Good, Fair, Marginal, Poor).
- Inspect multi-sensor correlation matrices and chemical interaction patterns.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add repository root to path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
%matplotlib inline
print("Environment initialized successfully!")

### 1. Ingest Data and Inspect Schema

In [ ]:
dataset_path = ROOT_DIR / "data" / "raw" / "Combined_dataset.csv"
# Load a representative subset for interactive EDA exploration
df = pd.read_csv(dataset_path, nrows=100000)
print(f"Loaded {len(df):,} rows with {df.shape[1]} columns.")
df.info()

In [ ]:
df.head()

### 2. Sensor Summary Statistics

In [ ]:
sensor_cols = [
    'Ammonia (mg/l)', 'Biochemical Oxygen Demand (mg/l)', 'Dissolved Oxygen (mg/l)',
    'Orthophosphate (mg/l)', 'pH (ph units)', 'Temperature (cel)',
    'Nitrogen (mg/l)', 'Nitrate (mg/l)'
]
df[sensor_cols].describe().T

### 3. Target Distribution Analysis (Continuous & Discrete)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Continuous target: CCME_Values
sns.histplot(df['CCME_Values'], kde=True, bins=30, ax=axes[0], color='#1f77b4')
axes[0].set_title('Continuous Target: CCME Values Distribution (0 - 100)', fontweight='bold')
axes[0].set_xlabel('CCME Values (WQI Score)')

# Discrete target: CCME_WQI
order = ['Poor', 'Marginal', 'Fair', 'Good', 'Excellent']
sns.countplot(data=df, x='CCME_WQI', order=order, ax=axes[1], palette='Blues_d')
axes[1].set_title('Discrete Target: CCME Quality Tiers', fontweight='bold')
axes[1].set_xlabel('Water Quality Class')

plt.tight_layout()
plt.show()

### 4. Waterbody Type Distribution

In [ ]:
plt.figure(figsize=(10, 5))
wb_counts = df['Waterbody Type'].value_counts()
sns.barplot(x=wb_counts.values, y=wb_counts.index, palette='viridis')
plt.title('Sample Frequency by Waterbody Type', fontweight='bold')
plt.xlabel('Count')
plt.show()

### 5. Multi-Sensor Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 8))
corr = df[sensor_cols + ['CCME_Values']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, cbar_kws={'label': 'Pearson r'})
plt.title('Sensor & Target Pearson Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

### Key Insights from EDA:
1. **Detection Limits:** Nutrients (Ammonia, Orthophosphate, Nitrate) exhibit minimum values of `0.0`, representing left-censored sensor detection limits.
2. **Sensor Physical Anomalies:** pH values exceed `14.0` (max = `19.44`), and temperatures approach boiling (>50°C), requiring physics bounds auditing.
3. **Target Distribution:** CCME_Values clusters heavily in the upper range (80-100, Good & Excellent), with a long tail into Poor (<45), requiring stratified evaluation.
4. **No Single Dominant Feature:** No individual sensor exhibits a Pearson $|r| > 0.8$ with the target, confirming that WQI prediction requires multi-parameter nonlinear modeling.